# Create merged df of all bid and most data

## Module load

In [ ]:
import pandas as pd
import re

Merge WTP data with main participant data

In [ ]:


# Define file paths
file1 = 'C:\\Users\\Patrick\\Gutbrain_R\\data\\2024-10-21GUTBRAINMAIN.csv'
file2 = 'C:\\Users\\Patrick\\Gutbrain_R\\data\\GB_bid_fmri.csv'

# Read the CSV files
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# Rename 'Session' to 'session' in df1
df1.rename(columns={'Session': 'session'}, inplace=True)

# Merge the DataFrames on 'id_participant' and 'session'
df = pd.merge(df1, df2, on=['id_participant', 'session'], how='left')



convert all to lower case

In [ ]:
 df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


Clean some columns for non fMRI analysis (behavioral only)

In [ ]:
# Columns to remove
columns_to_remove = [
    'record_id', 'ID participant + session + date irm non vide',
    'Le dossier existe sur CC', 'Reconciliation fichier suivi analyses SPM',
    'Date d\'irm ou pas?', 'dna', 'dna_chx',  'main_dominante', 'grandeur_recherche'
]

# Remove the specified columns
df = df.drop(columns=columns_to_remove)


Prepare and clean to include food stimuli info from participants (liking, etc.)

In [ ]:
# Path to the new CSV file
csv_file_path_choixcoll = 'C:\\Users\\Patrick\\Gutbrain_R\\data\\GUTBRAIN-ChoixDesCollationsLi_DATA_2024-10-23_1139.csv'

# Create a new DataFrame
ChoixColl = pd.read_csv(csv_file_path_choixcoll)

# Display the first few rows of the new DataFrame
ChoixColl.head()

# Replace 'skittes' with 'skittles' in the column names of ChoixColl
ChoixColl.columns = ChoixColl.columns.str.replace('skittes', 'skittles')

# Replace 'handisnack' with 'handisnacks' in the column names of ChoixColl
ChoixColl.columns = ChoixColl.columns.str.replace('handisnack', 'handisnacks')

# Replace 'handisnackss' with 'handisnacks' in the column names of ChoixColl
ChoixColl.columns = ChoixColl.columns.str.replace('handisnackss', 'handisnacks')

ChoixColl.columns = ChoixColl.columns.str.replace('_', '')

# Fill empty values in 'idrecherchescan' with values from 'q3'
ChoixColl['idrecherchescan'] = ChoixColl['idrecherchescan'].fillna(ChoixColl['q3'])

# clean some name typos from the redcap survey
df['picture'] = df['picture'].replace('tomatescerises', 'tomates')
df['picture'] = df['picture'].replace('juliennes', 'julienne')
df['picture'] = df['picture'].replace('cheesestrings', 'ficello')
df['picture'] = df['picture'].replace({
    'cucumber': 'concombre',
    'pretzels': 'roldgold',
    'applesauce': 'compote',
    'plum': 'prune',
    'apple': 'pomme',
    'mandm': 'mm',
    'banana': 'banane',
    'julienne' : 'juliennebbq',
    'verrycherry': 'delmonte',
    'oatmeal': 'gruau',
    'ruffles': 'rufflescremesure',
    'redpepper': 'poivron',
    'rufflesregular': 'rufflesnature',
    'm&mamandes': 'mmamandes',
    'lays': 'laysclassique',
    'maynards' : 'maynardframboise',
    'welchs': 'welch',
    'maynardspeaches': 'maynardpeches',
    'pear': 'poire',
    'melangerandonneur': 'krispykernel',
    'goldfish2': 'goldfish',
    'amandes': 'amandessalees'
})

# Rename the columns in the DataFrame
ChoixColl = ChoixColl.rename(columns={
    'reese1': 'reese1',
    'reese3': 'reese2',
    'reese4': 'reese3',
    'reese2': 'reese4'
})

# Convert column names of ChoixColl to lowercase for case-insensitive comparison
choixcoll_columns_lower = [col.lower() for col in ChoixColl.columns]



## validation of missing data

In [ ]:
donnees_manquante_df = df[['session', 'id_participant', 'age', 'picture','Commentaires']].copy()
donnees_manquante_df = donnees_manquante_df[(donnees_manquante_df['age'].isna()) | (donnees_manquante_df['picture'].isna())]
donnees_manquante_df = donnees_manquante_df[
    ~donnees_manquante_df['id_participant'].str.contains('rgc', na=False) & 
    donnees_manquante_df['session'].isin(['3', '4'])
]
donnees_manquante_df

# Save the donnees_manquante_df DataFrame to a CSV file
donnees_manquante_df.to_csv('C:\\Users\\Patrick\\Gutbrain_R\\data\\donnee_manquante_df.csv', index=False)

In [ ]:
unique_combinations = donnees_manquante_df[['session', 'id_participant']].drop_duplicates()
print(unique_combinations)
row_count = unique_combinations.shape[0]
print(f"Number of unique combinations: {row_count}")

drop participants with only one session of data

In [ ]:
# List of participants to drop (they only have session 1)
participants_to_drop = ['rnd066', 'rnd056', 'rnd008', 'rnd017']

# Drop rows where id_participant is in the list
df = df[~df['id_participant'].isin(participants_to_drop)]



Further cleaning

In [ ]:
df.loc[:, 'picture'] = df['picture'].str.lower()
f = df.drop(columns=['Donnees utilisee pour CMDO'])

Get ready to add snack ratings per participant

In [ ]:
# Insert new columns after the 'value' column
df.insert(df.columns.get_loc('value') + 1, 'known', None)
df.insert(df.columns.get_loc('value') + 2, 'freq', None)
df.insert(df.columns.get_loc('value') + 3, 'like', None)
df.insert(df.columns.get_loc('value') + 4, 'estcal', None)

# Create a list of column names
columns = list(ChoixColl.columns)

# Insert empty columns to the right of columns containing '4'
i = 0
while i < len(columns):
    if '4' in columns[i]:
        columns.insert(i + 1, f'empty_col_{i + 1}')
        i += 1  # Skip the newly added column
    i += 1

# Reindex the DataFrame with the new columns
ChoixColl = ChoixColl.reindex(columns=columns)

# Identify columns with 'empty' in their name
empty_columns = [col for col in ChoixColl.columns if 'empty' in col]

# Concatenate values from the 4 previous columns for each row
for empty_col in empty_columns:
    empty_col_index = ChoixColl.columns.get_loc(empty_col)
    if isinstance(empty_col_index, int) and empty_col_index >= 4:  # Ensure there are at least 4 previous columns
        previous_cols = ChoixColl.columns[empty_col_index-4:empty_col_index]
        ChoixColl[empty_col] = ChoixColl[previous_cols].astype(str).agg(';'.join, axis=1)

# Loop through the columns
for i in range(1, len(ChoixColl.columns)):
    if 'empty' in ChoixColl.columns[i]:
        previous_col = ChoixColl.columns[i - 1]
        if '4' in previous_col:
            new_name = previous_col.split('4')[0]
            ChoixColl.rename(columns={ChoixColl.columns[i]: new_name}, inplace=True)

# List to keep track of columns to drop
columns_to_drop = []

# Loop through columns
for i, col in enumerate(ChoixColl.columns):
    if '4' in col:
        # Add this column and the previous three columns to the drop list
        columns_to_drop.extend(ChoixColl.columns[max(0, i-3):i+1])

# Drop the columns and create a new DataFrame
ChoixCollConcat = ChoixColl.drop(columns=columns_to_drop)

# Replace column name 'idrecherchescan' with 'id_participant'
ChoixCollConcat.rename(columns={'idrecherchescan': 'id_participant'}, inplace=True)

# Melt the DataFrame 
ChoixCollConcat_melted = pd.melt(ChoixCollConcat, id_vars=['id_participant'], var_name='picture', value_name='value')

# Get unique values of 'picture' column from both dataframes
unique_pictures_df = df['picture'].unique()
unique_pictures_choixcollconcat_melted = ChoixCollConcat_melted['picture'].unique()

# Find the differences between the two sets of unique values
unique_in_df_not_in_choixcollconcat_melted = set(unique_pictures_df) - set(unique_pictures_choixcollconcat_melted)
unique_in_choixcollconcat_melted_not_in_df = set(unique_pictures_choixcollconcat_melted) - set(unique_pictures_df)

# Print the differences
print("Unique values in df['picture'] not in ChoixCollConcat_melted['picture']:", unique_in_df_not_in_choixcollconcat_melted)
print("Unique values in ChoixCollConcat_melted['picture'] not in df['picture']:", unique_in_choixcollconcat_melted_not_in_df)

# Check if there are exact matches
if not unique_in_df_not_in_choixcollconcat_melted and not unique_in_choixcollconcat_melted_not_in_df:
    print("There are exact matches for unique values of 'picture' column in both dataframes.")
else:
    print("There are differences in the unique values of 'picture' column between the two dataframes.")

ChoixCollConcat_melted = ChoixCollConcat_melted.applymap(lambda x: x.lower() if isinstance(x, str) else x)
ChoixCollConcat_melted

# Assuming ChoixCollConcat_melted and df are already defined

# Step 1: Extract the picture columns and strip spaces
pictures_choixcoll = ChoixCollConcat_melted['picture'].str.strip()
pictures_df = df['picture'].str.strip()

# Step 2: Find pictures in ChoixCollConcat_melted that are not in df
unmatched_pictures = pictures_choixcoll[~pictures_choixcoll.isin(pictures_df)]

# Step 3: Find unique unmatched pictures
unique_unmatched_pictures = unmatched_pictures.unique()

# Step 4: Print the unique unmatched pictures
print("Unique unmatched pictures in ChoixCollConcat_melted:")
print(unique_unmatched_pictures)

## Few validations of df quality + merge snack ratings and main df

In [ ]:
missing_keys_df = df[~df[['id_participant', 'picture']].isin(ChoixCollConcat_melted[['id_participant', 'picture']].to_dict(orient='list')).all(axis=1)]
print("Missing keys in df:")
print(missing_keys_df)

# Check for discrepancies in 'id_participant' and 'picture' columns
print("Discrepancies in 'id_participant':")
print(set(df['id_participant']).difference(set(ChoixCollConcat_melted['id_participant'])))
print("Discrepancies in 'picture':")
print(set(df['picture']).difference(set(ChoixCollConcat_melted['picture'])))


# Merge df and ChoixCollConcat_melted by 'id_participant' and 'picture'
merged_df = pd.merge(df, ChoixCollConcat_melted, on=['id_participant', 'picture'], how='left')

# Create a non-merged DataFrame
#non_merged_df = ChoixCollConcat_melted.merge(df, on=['id_participant', 'picture'], how='left', indicator=True)
#non_merged_df = non_merged_df[non_merged_df['_merge'] == 'left_only'].drop(columns=df.columns.difference(['id_participant', 'picture', 'value']))


In [ ]:
merged_df.rename(columns={'value_x': 'bid'}, inplace=True)
merged_df[['known', 'freq', 'like', 'estcal']] = merged_df['value_y'].str.split(';', expand=True)

In [ ]:
print(merged_df.columns)

In [ ]:
finaldf = merged_df

# Perform the inverse replacements on finaldf
finaldf['picture'] = finaldf['picture'].replace('tomates', 'tomatescerises')
finaldf['picture'] = finaldf['picture'].replace('julienne', 'juliennes')
finaldf['picture'] = finaldf['picture'].replace('ficello', 'cheesestrings')
finaldf['picture'] = finaldf['picture'].replace({
    'concombre': 'cucumber',
    'roldgold': 'pretzels',
    'compote': 'applesauce',
    'prune': 'plum',
    'pomme': 'apple',
    'mm': 'mandm',
    'banane': 'banana',
    'juliennebbq': 'julienne',
    'delmonte': 'verrycherry',
    'gruau': 'oatmeal',
    'rufflescremesure': 'ruffles',
    'poivron': 'redpepper',
    'rufflesnature': 'rufflesregular',
    'mmamandes': 'm&mamandes',
    'laysclassique': 'lays',
    'maynardframboise': 'maynards',
    'welch': 'welchs',
    'maynardpeches': 'maynardspeaches',
    'poire': 'pear',
    'krispykernel': 'melangerandonneur',
    'goldfish': 'goldfish2',
    'amandessalees': 'amandes'
})

## Prepare to add true caloric density to main df

In [ ]:
# Path to the CSV file
csv_file_path = 'C:\\Users\\Patrick\\Gutbrain_R\\data\\kcal_real_v3.csv'

# Load the CSV file into a DataFrame
dfkcalreal = pd.read_csv(csv_file_path)

# Convert the 'picture' column to lowercase in df
finaldf['picture'] = finaldf['picture'].str.lower()

# Convert the 'picture' column to lowercase in dfkcalreal
dfkcalreal['picture'] = dfkcalreal['picture'].str.lower()

# Insert the new column 'kcalreal' after the 'estcal' column
finaldf.insert(df.columns.get_loc('estcal') + 1, 'kcalreal', None)

dfkcalreal.rename(columns={'Moyenne Densite': 'kcalreal'}, inplace=True)

columns_to_remove = ['Dénomination', 'Différence', 'Source', 'Source autre', 'Annee Prix', 'Source Prix (si autre)', 'Pays prix', 'Commentaire']
dfkcalreal = dfkcalreal.drop(columns=columns_to_remove, errors='ignore')


Clean kcal real df (raw price and caloric density info)

In [ ]:
# Extract food category and food name from paths such as:
# food/hisweetmaynards.bmp -> HiSweet, maynards
# food/lonectarine.jpg -> Lo, nectarine
parsed_picture = dfkcalreal["picture"].str.extract(
    r"^food/(?P<type>hi(?:sweet|salt)|lo)(?P<picture>.+)\.(?:bmp|jpg)$",
    expand=True,
    flags=re.IGNORECASE
)

dfkcalreal[["type", "picture"]] = parsed_picture
dfkcalreal["type"] = dfkcalreal["type"].str.lower()
dfkcalreal["picture"] = dfkcalreal["picture"].str.lower()

Clean strings + validation

In [ ]:
# Canonicalize known picture-name variants used by the two datasets
picture_aliases = {
    "juliennes": "julienne",
}

# Rebuild the normalized picture values after applying aliases
final_picture_values = (
    finaldf["picture"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace(picture_aliases)
)

kcal_picture_values = (
    dfkcalreal["picture"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace(picture_aliases)
)

# Convert the refreshed values to sets
final_pictures = set(final_picture_values.dropna())
kcal_pictures = set(kcal_picture_values.dropna())

matching_unique = final_pictures & kcal_pictures

print(f"Matching unique pictures: {len(matching_unique)}")

print("Pictures in finaldf but not dfkcalreal:")
print(sorted(final_pictures - kcal_pictures))

print("Pictures in dfkcalreal but not finaldf:")
print(sorted(kcal_pictures - final_pictures))

assert "julienne" not in (final_pictures - kcal_pictures)
assert "julienne" not in (kcal_pictures - final_pictures)

Merge main DF and caloric density data

In [ ]:
finalmerged_df = finaldf.merge(dfkcalreal, on='picture', how='left')


Clean current version of the final df

In [ ]:
columns_to_drop = ['value_y','kcalreal_x']
finalmerged_df.drop(columns=columns_to_drop, inplace=True)

Get ready to merge psychological questionnaires data

In [ ]:
# Define the file path
csv_file_path = 'C:\\Users\\Patrick\\Gutbrain_R\\data\\bdd_questionnaires_results.csv'

# Read the CSV file into a DataFrame
df_quest = pd.read_csv(csv_file_path)



Clean psychological questionnaires data

In [ ]:
df_quest.rename(columns={'Sbj_ID': 'id_participant'}, inplace=True)
df_quest['id_participant'] = df_quest['id_participant'].str.lower()
df_quest.drop(columns=['redcap_survey_identifier'], inplace=True)



Merge with main df

In [ ]:
final_merged_quest = pd.merge(finalmerged_df, df_quest, on=['id_participant', 'session'], how='left')


Prepare main df for further analysis

In [ ]:
# Create a new column 'sexeF' based on the 'sexe' column
final_merged_quest['sexeF'] = final_merged_quest['sexe'].apply(lambda x: 1 if x == 2 else 0)

# Insert the 'sexeF' column beside the 'sexe' column
sexe_index = final_merged_quest.columns.get_loc('sexe')
final_merged_quest.insert(sexe_index + 1, 'sexeF', final_merged_quest.pop('sexeF'))

# Change the format of 'type_chx' to a string
final_merged_quest['type_chx'] = final_merged_quest['type_chx'].astype(str)






save the final DF

In [ ]:
# Save the DataFrame to a CSV file
final_merged_quest.to_csv('C:\\Users\\Patrick\\Gutbrain_R\\data\\final_merged_quest.csv', index=False)